In [1]:
import json
import torch
import os
import sys
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from langchain_community.embeddings import HuggingFaceEmbeddings, HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS


# os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

dirs = ["..", 'create_prompt_llm/']
for _dir in dirs:
    if _dir not in sys.path:
        sys.path.append(_dir)

import covmis, liar2, prompt_rag
data_test = covmis.load_data('test')
data_entire = covmis.load_data('entire')
data_search = covmis.load_data_search('entire', None)

from swift.llm import (
    ModelType, get_vllm_engine, get_default_template_type,
    get_template, inference_vllm, VllmGenerationConfig
)
from custom import CustomModelType

model_type = 'llama3_1-70b-instruct-awq' # CustomModelType.llama_3_70b_instruct_awq

model_kwargs = {'device': 'cuda:2'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
# query_instruction_for_rag = "Represent this sentence for searching relevant passages: "

# embedding_mxbai = HuggingFaceEmbeddings(
#     model_name='/home/css/models/mxbai-embed-large-v1',
#     model_kwargs=model_kwargs,
#     encode_kwargs=encode_kwargs,
# )

# vector_store_norag = FAISS.load_local(
#     "/home/hanlv/workspace/code/research/infodemic/LLM/lanchain/vector_store/db_faiss_related_articles_mxbai", 
#     embedding_mxbai, allow_dangerous_deserialization=True)

llm_engine = get_vllm_engine(
    model_type, 
    torch_dtype=torch.float16,  # 检查正确的数据类型！！！！
    tensor_parallel_size=2,
    max_model_len=4096,
    gpu_memory_utilization=0.92,
    model_id_or_path="/home/css/models/Meta-Llama-3.1-70B-Instruct-AWQ-INT4",
    max_num_seqs=32,
    engine_kwargs = {
        # "enforce_eager": True,
        "seed": 42,
    }
)

template_type = get_default_template_type(model_type)
template = get_template(template_type, llm_engine.hf_tokenizer)

generation_config = VllmGenerationConfig(
    max_tokens=1,
    temperature=0,
    seed=42,
    logprobs=20
)

get_resp_list = lambda request_list : inference_vllm(
    llm_engine, template, request_list, 
    generation_config=generation_config, 
    use_tqdm=True, 
)



[INFO:swift] Successfully registered `/home/hanlv/workspace/code/research/infodemic/LLM/swift/swift/llm/data/dataset_info.json`
[INFO:swift] No LMDeploy installed, if you are using LMDeploy, you will get `ImportError: cannot import name 'prepare_lmdeploy_engine_template' from 'swift.llm'`
[INFO:swift] Loading the model using model_dir: /home/css/models/Meta-Llama-3.1-70B-Instruct-AWQ-INT4


INFO 11-06 19:40:06 awq_marlin.py:90] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 11-06 19:40:06 config.py:899] Defaulting to use mp for distributed inference
INFO 11-06 19:40:06 llm_engine.py:226] Initializing an LLM engine (v0.6.1.dev238+ge2c6e0a82) with config: model='/home/css/models/Meta-Llama-3.1-70B-Instruct-AWQ-INT4', speculative_config=None, tokenizer='/home/css/models/Meta-Llama-3.1-70B-Instruct-AWQ-INT4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=awq_marlin, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observab

Loading safetensors checkpoint shards:   0% Completed | 0/9 [00:00<?, ?it/s]


(VllmWorkerProcess pid=1966561) INFO 11-06 19:40:23 model_runner.py:1025] Loading model weights took 18.5809 GB
INFO 11-06 19:40:23 model_runner.py:1025] Loading model weights took 18.5809 GB
INFO 11-06 19:40:27 distributed_gpu_executor.py:57] # GPU blocks: 875, # CPU blocks: 1638
INFO 11-06 19:40:29 model_runner.py:1329] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 11-06 19:40:29 model_runner.py:1333] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
(VllmWorkerProcess pid=1966561) INFO 11-06 19:40:29 model_runner.py:1329] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in

In [3]:
import numpy as np
K = 5

def get_prompt_with_nothing(item):
    claim = item['claim']
    claim_date = item['date']
    pre = 'Please answer TRUE or FALSE. Below is a <CLAIM>. If the content described by the <CLAIM> is correct, then classify it as TRUE; if the content described by the <CLAIM> is incorrect, then classify it as FALSE.\n\n'
    text = f"<CLAIM>: {claim}\nPublication date: {claim_date}"

    return pre + text

def get_prompt_with_OS(item, search_results):
    claim = item['claim']
    claim_date = item['date']
    pre = 'Please answer TRUE or FALSE. Below is some INFORMATION searched online and a <CLAIM>. Please classify the <CLAIM> as TRUE or FALSE based on the INFORMATION. If the content described by the <CLAIM> is correct, then classify it as TRUE; if the content described by the <CLAIM> is incorrect, then classify it as FALSE.\n\n'
    
    ids = slice(0, K)
    snippet = prompt_rag.get_brave_snippet(search_results, ids=ids)
    info = "INFORMATION:\n" + snippet + '\n'
    text = f"<CLAIM>: {claim}\nPublication date: {claim_date}"

    # print(pre + info + text)
    return pre + info + text

# env: torch 
# def get_prompt_with_RA(item):
#     claim = item['claim']
#     claim_date = item['date']
#     pre = 'Please answer TRUE or FALSE. Below is some KNOWN INFORMATION and a <CLAIM>. Please classify the <CLAIM> as TRUE or FALSE based on the KNOWN INFORMATION. If the content described by the <CLAIM> is correct, then classify it as TRUE; if the content described by the <CLAIM> is incorrect, then classify it as FALSE.\n\n'

#     docs = vector_store_norag.similarity_search(claim, k=K)
#     context = [doc.page_content for doc in docs]

#     context_new = ""
#     for i, item in enumerate(context):
#         context_new += f"{i+1}. " + item + '\n'

#     info = "KNOWN INFORMATION:\n" + context_new + '\n'
#     text = f"<CLAIM>: {claim}\nPublication date: {claim_date}"

#     # print(pre + info + text)
#     return pre + info + text


def get_label(response):
    """
    response -> label:

    TRUE. -> 2 and FALSE. -> 0

    """
    # 0:false, 2:true

    if response == "TRUE.":
        return 2
    elif response == "FALSE.":
        return 0
    else:
        raise Exception("Response既不是'TRUE.'，也不是'FALSE.'。")
        
    
def get_ans(logprobs):
    p_true = -np.inf
    p_false = -np.inf

    TRUE = 21260
    FALSE = 31451
    logprob_true = logprobs.get(TRUE)
    logprob_false = logprobs.get(FALSE)
    if logprob_true is None and logprob_false is None:
        return None, -np.inf
    if logprob_true is not None:
        p_true = logprob_true.logprob
    if logprob_false is not None:
        p_false = logprob_false.logprob
    
    if p_true > p_false:
        return "TRUE.", p_true
    else:
        return "FALSE.", p_false

def find_ind(idd):
    for i in range(len(data_entire)):
        if data_entire[i]["id"] == idd:
            return i
    return None


In [4]:
request_list = []
labels = []
preds = []
logprobs = []

key_label = 'label2'
for i, item in enumerate(data_test):
    # if int(item[key_label]) == 1:
    #     continue
    request_list.append({'query': get_prompt_with_OS(
        item, data_search[find_ind(item["id"])]["brave_search_results"]
    )})
    labels.append(item[key_label])

resp_list = inference_vllm(
    llm_engine, template, request_list,
    generation_config=generation_config, 
    use_tqdm=True, 
)

cnt = 0
for i, resp in enumerate(resp_list):
    # print(f"query: {request['query']}")
    ans, logprob = get_ans(resp['logprobs'][0])
    if ans is None:
        cnt += 1
        if labels[i] == 0:
            preds.append(2)
        else:
            preds.append(0)
    else:
        preds.append(get_label(ans))
    if np.isinf(logprob):
        logprobs.append(-99999)
    else:
        logprobs.append(logprob)
    # print(f"response: {resp['response']}")

metric = (
    accuracy_score(labels, preds), 
    f1_score(labels, preds, average='macro'),
    precision_score(labels, preds, average='macro'), 
    recall_score(labels, preds, average='macro'))
with_or_without_info = 'without_info'   # 'without_info', 'with_llama3_info', 'with_info'

print(metric)
print(cnt)


100%|██████████| 1439/1439 [20:37<00:00,  1.16it/s]

(0.8957609451007644, 0.8661305687450694, 0.8388463560789433, 0.9184631101656964)
0


In [ ]:
# data_preds = []

# for i, item in enumerate(data_test):
#     if int(item["label"]) == 1:
#         continue
#     item["pred"] = preds[i]
#     item["logprob"] = logprobs[i]

# with open(f'test_metric_no_sft/covmis/test/{with_or_without_info}/data_Meta-Llama-3-70B-Instruct(_llama3).json', 'r') as f:
#     data_preds = json.load(f)

# with open(f'test_metric_no_sft/covmis/test/{with_or_without_info}/data_Meta-Llama-3-70B-Instruct(_llama3).json', 'w') as f:
#     json.dump(data_preds, f, indent=4)


FileNotFoundError: [Errno 2] No such file or directory: 'test_metric_no_sft/covmis/test/without_info/data_Meta-Llama-3-70B-Instruct(_llama3).json'

In [ ]:
# with_or_without_info = 'without_info'
# import json
# import numpy as np
# with open(f'/home/hanlv/workspace/code/research/infodemic/LLM/swift/examples/pytorch/llm/my_inferencing/test_metric_single_llm/covmis/test/without_info/data_Meta-Llama-3-70B-Instruct(_llama3).json', 'r') as f:
#     data_preds = json.load(f)

# error_max = -9999
# ok_min = 0
# cnt = 0
# for item in data_preds:

#     if item["label"] != item["pred"]:
#         error_max = max(error_max, item["logprob"])
#     else:
#         ok_min = min(ok_min, item["logprob"])
#     if np.exp(item["logprob"]) > 0.1:
#         cnt += 1
# np.exp(error_max), np.exp(ok_min), cnt


(0.9392042587046251, 0.0008160251456512921, 826)

In [11]:
with_or_without_info = 'with_llama3_1_info'   # 'without_info', 'with_llama3_info', 'with_info', 'with_llama3_1_info'

def add_metrics(metrics, metric):
    metrics.append({
            "model": "Meta-Llama-3-70B-Instruct",
            "ACC": metric[0],
            "F1": metric[1],
            "Precision": metric[2],
            "Recall": metric[3]
        }
    )

try:
    with open(f'test_metric_no_sft/covmis2024/test/{with_or_without_info}/brave/Meta-Llama-3_1-70B-Instruct.json', 'r') as f:
        metrics = json.load(f)
except Exception as e:
    if isinstance(e, FileNotFoundError):
        metrics = []
    else:
        raise Exception(e)
    
add_metrics(metrics, metric)
with open(f'test_metric_no_sft/covmis2024/test/{with_or_without_info}/brave/Meta-Llama-3_1-70B-Instruct.json', 'w') as f:
    json.dump(metrics, f, indent=4)
